In [0]:
from pyspark.sql import functions as F

orders = spark.table("workspace.bronze.orders")

# Add validation flags
validated_orders = (
    orders
    .withColumn(
        "dq_error",
        F.when(F.col("order_id").isNull(), "Missing order_id")
         .when(F.col("customer_id").isNull(), "Missing customer_id")
         .when(F.col("total_amount") < 0, "Negative total_amount")
         .when(
             ~F.col("order_status").isin(
                 "COMPLETED",
                 "SHIPPED",
                 "PROCESSING",
                 "CANCELLED"
             ),
             "Invalid order_status"
         )
    )
)

# Valid records
valid_orders = (
    validated_orders
    .filter(F.col("dq_error").isNull())
    .drop("dq_error")
)

# Invalid records
bad_orders = (
    validated_orders
    .filter(F.col("dq_error").isNotNull())
)

display(valid_orders)
display(bad_orders)

In [0]:
valid_orders.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.silver.valid_orders")

bad_orders.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.silver.bad_orders")